In [1]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate
from langchain.prompts.example_selector import LengthBasedExampleSelector
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.messages import SystemMessage
import tiktoken

In [2]:
examples = [
    {"input": "Gollum", "output": "<Story involving Gollum>"},
    {"input": "Gandalf", "output": "<Story involving Gandalf>"},
    {"input": "Bilbo", "output": "<Story involving Bilbo>"},
]

In [3]:
story_prompt = PromptTemplate(
    input_variables=["input", "output"],
    template="Character: {input}\nStory: {output}",
)

In [4]:
def num_tokens_from_string(string: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding("cl100k_base")
    num_tokens = len(encoding.encode(string))
    return num_tokens

In [5]:
example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=story_prompt,
    max_length=1000, # 1000 tokens are to be included from examples in the prompt
    # get_text_length: Callable[[str], int] = lambda x: len(re.split("\n| ", x))
    # You have modified the get_text_length function to work with the TikToken library based on token usage:
    get_text_length=num_tokens_from_string,
)

In [6]:
dynamic_prompt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=story_prompt,
    prefix="Generate a story for {character} using the current Character/Story pairs from all of the characters as context.",
    suffix="Character: {character}\nStory:",
    input_variables=["character"],
)

In [7]:
# Provide a new character from lord of the rings
formatted_prompt = dynamic_prompt.format(character="Frodo")

In [8]:
print(formatted_prompt)

Generate a story for Frodo using the current Character/Story pairs from all of the characters as context.

Character: Gollum
Story: <Story involving Gollum>

Character: Gandalf
Story: <Story involving Gandalf>

Character: Bilbo
Story: <Story involving Bilbo>

Character: Frodo
Story:


In [9]:
model = ChatOpenAI()

result = model.invoke([SystemMessage(content=formatted_prompt)])

print(result.content)

Once upon a time in the land of Middle-earth, Frodo Baggins found himself entrusted with a great and dangerous task. His uncle, Bilbo Baggins, had left behind a powerful ring that held the fate of the world in its grasp. With the help of his wise mentor, Gandalf the Grey, Frodo learned of the true nature of the ring and the dark forces that sought to claim it.

As Frodo set out on his journey to destroy the ring, he encountered a strange and twisted creature known as Gollum. Gollum had once possessed the ring and its corrupting power had driven him to madness. Despite Gollum's deceitful nature, Frodo saw a glimmer of the creature's former self and felt a pang of pity for him.

Throughout his perilous quest, Frodo faced countless challenges and temptations, but he remained steadfast in his determination to destroy the ring and save Middle-earth from the evil that threatened to consume it. With the guidance of Gandalf, the wisdom of Bilbo, and even the unlikely help of Gollum, Frodo pers